# AB Dor Visibility-Model Fit

Joint instrument + kernel-visibility fit of AB Dor and its calibrators from calslope data, modeled on `wr137_vis.ipynb` but without WR137's `MonthVisFit`/`WheelVisFit` time-binning subclasses -- AB Dor's calslope data is a single visit, so plain `amigo.model_fits.SplineVisFit` exposures are used directly. Updated for `amigo`'s `retrain` branch and the `v_0.0.11` amigo files cache. No `dorito` dependency.

This fits `amigo.vis_models.LogVisModel` amplitude/phase basis coefficients jointly with the instrument model (aberrations, positions, fluxes, spectra) across all AB Dor + calibrator exposures. The resulting per-exposure amplitudes/phases are the calibrated kernel visibilities that feed into an OI-based reconstruction (e.g. `ab-dor.ipynb`).

**Note:** fixing this against dLux 0.15 required a one-line fix in `amigo/src/amigo/vis_models.py` (`wf_fft_coords` was assuming `wfs.pixel_scale` is a scalar; under dLux 0.15 it's a per-wavelength vector, constant in value).

In [1]:
# jax ecosystem
import jax

jax.config.update("jax_enable_x64", True)

device = jax.local_devices()[0]
print(device.device_kind)

from jax import numpy as np, random as jr, tree as jtu
import os

from amigo.fitting import sgd, adam  # moved from zodiax.optimisation

import amigo

# visualisation
import matplotlib.pyplot as plt
import matplotlib as mpl
import ehtplot
import scienceplots

plt.style.use(["science", "bright", "no-latex"])
plt.rcParams["image.cmap"] = "inferno"
plt.rcParams["font.family"] = "serif"
plt.rcParams["image.origin"] = "lower"
plt.rcParams["figure.dpi"] = 300
plt.rcParams["font.size"] = 8
plt.rcParams["xtick.direction"] = "out"
plt.rcParams["ytick.direction"] = "out"

inferno = mpl.colormaps["inferno"]
viridis = mpl.colormaps["viridis"]
seismic = mpl.colormaps["seismic"]
coolwarm = mpl.colormaps["coolwarm"]

inferno.set_bad("k", 0.5)
viridis.set_bad("k", 0.5)
seismic.set_bad("k", 0.5)
coolwarm.set_bad("k", 0.5)

cpu


In [2]:
from socket import gethostname

if gethostname() == "glinton":
    morgana = "/media/morgana1/"
else:
    morgana = "/Volumes/morgana1/"

data_path = os.path.join(morgana, "snert/max/data/JWST/AB-DOR/calslope/")
uncal_path = os.path.join(morgana, "snert/max/data/JWST/AB-DOR/uncal/")
amigo_cache = os.path.join(morgana, "snert/max/data/amigo_files/")

cache = os.path.join(amigo_cache, "v_0.0.11/")
output_path = os.path.join(amigo_cache, "outputs/AB-DOR/")

n_files_per_filt = 1  # max out at 25 (HD-36805's F480M dither sequence)
one_filt_flag = True  # if True, only use F480M
pos1_only = False
n_epoch = 200
save_path = None
# intermediate_prints = []
intermediate_prints = [25, 50, 100, 150]
intermediate_prints = [e for e in intermediate_prints if e < n_epoch]

# TARGPROP values in the data: science target is "AB-DOR", PSF calibrators are
# "HD-37093" and "HD-36805" (see chromatic_psf.ipynb).
EXP_TYPE = "NIS_AMI"
FILTERS = [
    "F480M",
    "F430M",
    "F380M",
]

# Bind file path, type and exposure type
file_fn = lambda data_path, filters=FILTERS, **kwargs: amigo.files.get_files(
    data_path,
    "calslope",
    EXP_TYPE=EXP_TYPE,
    FILTER=filters,
    **kwargs,
)

In [3]:
import importlib

# Same shared bad-pixel map used in retrain.ipynb
resource_path = importlib.resources.files(amigo).joinpath("data/badpix.npy")
with importlib.resources.as_file(resource_path) as file_path:
    badpix = np.array(np.load(file_path), dtype=int)
badpix_bool = badpix.astype(
    bool
)  # use for boolean masking; badpix (int) is kept for FITS writing


def add_badpix(file):
    file["BADPIX"].data = badpix
    if file[0].header["EXP_TYPE"] == "NIS_DARK":
        file["BADPIX"].data[25, 74] = 1
    return file

In [4]:
from amigo.misc import get_pos

# Same file-sorting convention as retrain.ipynb: prefer POS1 dithers (falling
# back to interleaved POS1/POS2), grouped by filter, truncated to at most `n`
# files per filter.
# Quick toggle flags -- controls how much data goes into the fit.


def sort_files(files, n):
    files.sort(
        key=lambda x: (
            x[0].header["PROGRAM"],
            x[0].header["FILTER"],
            x[0].header["PATT_NUM"],
        ),
        reverse=True,
    )
    filts = ["F480M", "F430M", "F380M"]
    these_files = {}

    for filt in filts:
        if one_filt_flag and filt != "F480M":
            these_files[filt] = []
            continue

        matched = [f for f in files if f[0].header["FILTER"] == filt]

        if pos1_only:
            x = [f for f in matched if get_pos(f) == "POS1"]
        else:
            pos1 = [f for f in matched if get_pos(f) == "POS1"]
            pos2 = [f for f in matched if get_pos(f) == "POS2"]
            x = []
            for a, b in zip(pos1, pos2):
                x.extend([a, b])
            x.extend(pos1[len(pos2) :])
            x.extend(pos2[len(pos1) :])

        these_files[filt] = x[0:n]

    files = [f for sub_f in these_files.values() for f in sub_f]
    return files

# Loading in data

In [5]:
from retrain_fns import summarise_files

# The known PSF calibrators for this program (see chromatic_psf.ipynb). A
# handful of single-frame acquisitions for other targets (e.g.
# "J062802.01-663738.0") also show up as IS_PSF=True in this folder; their
# TARGPROP contains a literal "." which breaks amigo's dotted param-key
# lookup, so they're excluded explicitly rather than accepted via IS_PSF alone.
CALIBRATORS = ["HD-37093", "HD-36805"]

sci_files = sort_files(file_fn(data_path, IS_PSF=False), n=n_files_per_filt)
cal_files = sort_files(file_fn(data_path, IS_PSF=True), n=n_files_per_filt)
cal_files = [f for f in cal_files if f[0].header["TARGPROP"] in CALIBRATORS]

sci_files = [add_badpix(f) for f in sci_files]
cal_files = [add_badpix(f) for f in cal_files]

summarise_files(sci_files)
summarise_files(cal_files)

  program  target filter dither   g/i        date                 PI    CAL
0    1093  AB-DOR  F480M    5/5  5/69  05-06-2022  Thatte, Deepashri  False
  program    target filter dither   g/i        date                 PI   CAL
0    1093  HD-36805  F480M  25/25  9/65  05-06-2022  Thatte, Deepashri  True


# Building the model

In [6]:
load_dict = lambda x: np.load(f"{x}", allow_pickle=True).item()

state = load_dict(cache + "calibration.npy")

cal_exposures = [amigo.model_fits.SplineVisFit(file) for file in cal_files]
sci_exposures = [amigo.model_fits.SplineVisFit(file) for file in sci_files]
exposures = cal_exposures + sci_exposures

model = amigo.core_models.AmigoModel(
    exposures,
    optics=amigo.optical_models.AMIOptics(sparse=True, radial_orders=4),
    detector=amigo.detector_models.LinearDetector(),
    ramp_model=amigo.ramp_models.NonLinearRamp(),
    read=amigo.read_models.ReadModel(),
    vis_model=amigo.vis_models.LogVisModel(load_dict(cache + "vis_basis.npy")),
    state=state,
)

TimeoutError: [Errno 60] Operation timed out: '/Volumes/morgana1/snert/max/data/amigo_files/v_0.0.11/vis_basis.npy'

In [ ]:
for exp in exposures:
    print(exp.get_key("aberrations"))
    exp.print_summary()
    amigo.plotting.summarise_fit(model, exp, residuals=False)

# Optimisation

In [ ]:
def summarise_fn(result, save_path):
    losses = np.array([v for v in result.losses.values()]).mean(0)
    amigo.plotting.plot_losses(
        losses, start=int(0.8 * len(losses)), save_path=save_path
    )
    amigo.plotting.plot(result.history, save_path=save_path)

    for exp in exposures:
        exp.print_summary()
        amigo.plotting.summarise_fit(result.model, exp, save_path=save_path)


# NOTE: untuned starting points, carried over from wr137_vis.ipynb -- check
# against the loss curve / residuals below and adjust for AB Dor's SNR.
config = {
    "positions": sgd(2e-1, 0),
    "fluxes": sgd(2e-1, 0),
    "aberrations": sgd(3e-2, 4),
    "spectra": sgd(2e-1, 10),
    # "phases": sgd(2e0, 20),
    # "amplitudes": sgd(2e0, 20),
}

trainer = amigo.fitting.Trainer(
    cache=os.path.join(amigo_cache, "fishers/"),
    summarise_fn=summarise_fn,
    intermediate_prints=intermediate_prints,
    save_path=save_path,
)

trainer = trainer.populate_fishers(
    model,
    exposures,
    hessians=load_dict(cache + "jacobians.npy")["hessian"],
    parameters=list(config.keys()),
)

print("Number of exposures: ", len(exposures))

# Train the model
result = trainer.train(
    model=model,
    optimisers=config,
    epochs=n_epoch,
    # AB Dor has far fewer exposures than WR137, so a single batch is fine to
    # start with -- switch to amigo.fitting.batch_exposures(exposures, n_batch=...)
    # if that gets too slow/memory-heavy.
    batches=exposures,
)

In [ ]:
summarise_fn(result, save_path)